# 01 · Data Exploration

Exploratory data analysis of the Ames Housing dataset used by this project. This notebook mirrors the logic in `src/data_loader.py` and `src/eda.py` — no duplicated preprocessing logic is written here.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src import config, data_loader, eda

pd.set_option('display.max_columns', 30)

## Load and clean the raw dataset

In [2]:
df = data_loader.load_raw_data()
df = data_loader.basic_clean(df)
df.shape

(2930, 82)

In [3]:
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,...,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,...,210,62,0,0,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Feedr,...,140,0,0,0,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,...,393,36,0,0,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,AllPub,Corner,Gtl,NAmes,Norm,...,0,0,0,0,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,...,212,34,0,0,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


## Dataset summary

Shape, dtypes, missing values, duplicates, skewness.

In [4]:
summary = eda.summarize_dataset(df)
summary

{'shape': (2930, 82),
 'n_duplicates': 0,
 'missing_values_top10': {'Pool QC': 2917,
  'Misc Feature': 2824,
  'Alley': 2732,
  'Fence': 2358,
  'Mas Vnr Type': 1775,
  'Fireplace Qu': 1422,
  'Lot Frontage': 490,
  'Garage Qual': 159,
  'Garage Yr Blt': 159,
  'Garage Cond': 159},
 'numeric_columns': 39,
 'categorical_columns': 43,
 'target_skewness': 1.744}

In [5]:
df.dtypes.value_counts()

str        43
int64      28
float64    11
Name: count, dtype: int64

In [6]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0].head(15)

Pool QC           2917
Misc Feature      2824
Alley             2732
Fence             2358
Mas Vnr Type      1775
Fireplace Qu      1422
Lot Frontage       490
Garage Qual        159
Garage Yr Blt      159
Garage Cond        159
Garage Finish      159
Garage Type        157
Bsmt Exposure       83
BsmtFin Type 2      81
Bsmt Qual           80
dtype: int64

## Target variable: SalePrice

The target is right-skewed, as is typical for house prices — most homes cluster in a moderate price range with a long tail of expensive properties.

In [7]:
df[config.TARGET_COLUMN].describe()

count      2930.000000
mean     180796.060068
std       79886.692357
min       12789.000000
25%      129500.000000
50%      160000.000000
75%      213500.000000
max      755000.000000
Name: SalePrice, dtype: float64

In [8]:
eda.plot_target_distribution(df, config.FIGURES_DIR)
print('Saved to', config.FIGURES_DIR / 'target_distribution.png')

Saved to /home/claude/house-price-prediction/reports/figures/target_distribution.png


![target distribution](../reports/figures/target_distribution.png)

## Correlations with key numeric features

In [9]:
eda.plot_correlation_heatmap(df, config.FIGURES_DIR)
print('Saved to', config.FIGURES_DIR / 'correlation_heatmap.png')

Saved to /home/claude/house-price-prediction/reports/figures/correlation_heatmap.png


![correlation heatmap](../reports/figures/correlation_heatmap.png)

## Price vs. living area, bedrooms, bathrooms, age, and location

In [10]:
eda.plot_price_vs_living_area(df, config.FIGURES_DIR)
eda.plot_price_vs_bedrooms(df, config.FIGURES_DIR)
eda.plot_price_vs_bathrooms(df, config.FIGURES_DIR)
eda.plot_price_vs_property_age(df, config.FIGURES_DIR)
eda.plot_price_by_location(df, config.FIGURES_DIR)
eda.plot_boxplots_numeric(df, config.FIGURES_DIR)
print('Saved all remaining EDA figures to', config.FIGURES_DIR)

Saved all remaining EDA figures to /home/claude/house-price-prediction/reports/figures


![price vs living area](../reports/figures/price_vs_living_area.png)
![price vs bedrooms](../reports/figures/price_vs_bedrooms.png)
![price vs bathrooms](../reports/figures/price_vs_bathrooms.png)
![price vs property age](../reports/figures/price_vs_property_age.png)
![price by location](../reports/figures/price_by_location.png)
![boxplots](../reports/figures/boxplots_numeric_features.png)

## Takeaways

- `Overall Qual`, `Gr Liv Area`, `Garage Cars`, and total bathrooms show the strongest visual relationship with `SalePrice`.
- `SalePrice` is right-skewed; tree-based models handle this natively, while linear models could benefit from a log-transformed target (left as a future improvement).
- Several columns (`Pool QC`, `Misc Feature`, `Alley`, `Fence`) are mostly missing because the feature is genuinely absent for most homes (e.g. no pool) — these columns are intentionally excluded from the curated feature set used for modeling.